In [ ]:
# Importação das bibliotecas principais
# TensorFlow será utilizado para Deep Learning
# TensorFlow Datasets será usado para carregar o dataset Cats vs Dogs
# Matplotlib será usado para visualização dos resultados

import tensorflow as tf
import tensorflow_datasets as tfds
import matplotlib.pyplot as plt


In [ ]:
# Carregamento do dataset Cats vs Dogs
# O dataset é dividido em 80% para treino e 20% para validação
# as_supervised=True retorna pares (imagem, rótulo)

(train_ds, val_ds), metadata = tfds.load(
    'cats_vs_dogs',
    split=['train[:80%]', 'train[80%:]'],
    with_info=True,
    as_supervised=True
)

# Verificando as informações do dataset
print(metadata)


In [ ]:
# Definição do tamanho das imagens
# MobileNetV2 trabalha bem com imagens 160x160
IMG_SIZE = (160, 160)

# Definição do tamanho do batch
BATCH_SIZE = 32

# Função de pré-processamento
# Redimensiona as imagens
# Normaliza os valores dos pixels para o intervalo [0,1]

def preprocess(image, label):
    image = tf.image.resize(image, IMG_SIZE)
    image = tf.cast(image, tf.float32) / 255.0
    return image, label

# Aplicando o pré-processamento
# batch(): agrupa os dados
# prefetch(): melhora o desempenho durante o treinamento

train_ds = train_ds.map(preprocess).batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)
val_ds = val_ds.map(preprocess).batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)


In [ ]:
# Carregamento do modelo MobileNetV2
# include_top=False remove a camada final original
# weights='imagenet' utiliza pesos pré-treinados

base_model = tf.keras.applications.MobileNetV2(
    input_shape=(160, 160, 3),
    include_top=False,
    weights='imagenet'
)

# Congelamento das camadas convolucionais
# Isso garante que apenas as camadas finais serão treinadas

base_model.trainable = False


In [ ]:
# Construção do modelo final
# GlobalAveragePooling reduz a dimensionalidade
# Dense adiciona camadas totalmente conectadas
# A última camada usa sigmoid para classificação binária

model = tf.keras.Sequential([
    base_model,
    tf.keras.layers.GlobalAveragePooling2D(),
    tf.keras.layers.Dense(128, activation='relu'),
    tf.keras.layers.Dense(1, activation='sigmoid')
])

# Exibindo o resumo do modelo
model.summary()


In [ ]:
# Compilação do modelo
# Adam é um otimizador eficiente
# Binary Crossentropy é usada para classificação binária
# Accuracy será usada como métrica de avaliação

model.compile(
    optimizer=tf.keras.optimizers.Adam(),
    loss='binary_crossentropy',
    metrics=['accuracy']
)


In [ ]:
# Treinamento do modelo
# O modelo será treinado por 5 épocas
# Os dados de validação são usados para avaliar a generalização

history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=5
)


529/582 ━━━━━━━━━━━━━━━━━━━━ 36s 687ms/step - accuracy: 0.9825 - loss: 0.0489

In [ ]:
# Extração dos valores de acurácia e perda
acc = history.history['accuracy']
val_acc = history.history['val_accuracy']

loss = history.history['loss']
val_loss = history.history['val_loss']

# Criação dos gráficos
plt.figure(figsize=(12,4))

# Gráfico de acurácia
plt.subplot(1,2,1)
plt.plot(acc, label='Treino')
plt.plot(val_acc, label='Validação')
plt.title('Acurácia')
plt.legend()

# Gráfico de loss
plt.subplot(1,2,2)
plt.plot(loss, label='Treino')
plt.plot(val_loss, label='Validação')
plt.title('Loss')
plt.legend()

plt.show()
